# SimCLR (NT-Xent) — Paper-to-Code Mock (Colab)

**Paper:** A Simple Framework for Contrastive Learning of Visual Representations (Chen et al., 2020) — https://arxiv.org/abs/2002.05709

Medium-hard mock (~40 min). Fill in the `nt_xent_loss` stub, run the no-labels contrastive toy task, then the sanity checks. Reference solution in the last cell.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. Implement the NT-Xent / InfoNCE loss (YOUR TASK)

Given `z1, z2` of shape `(N, D)` (projection-head outputs of the two views):
1. Stack into `(2N, D)` and **L2-normalize** (so dot products = cosine sim).
2. Build the `2N x 2N` similarity matrix divided by `temperature`.
3. **Mask the diagonal** to `-inf` (a view can't match itself).
4. Cross-entropy where row `i`'s positive is its paired view (`i+N` / `i-N`).

In [ ]:
def nt_xent_loss(z1, z2, temperature=0.5):
    N = z1.shape[0]
    # TODO: z = cat([z1, z2]); L2-normalize
    # TODO: sim = z @ z.t() / temperature
    # TODO: mask diagonal (self-similarity) to -inf
    # TODO: targets = [N..2N-1, 0..N-1]; return F.cross_entropy(sim, targets)
    raise NotImplementedError("fill me in")


class Encoder(nn.Module):
    """Small MLP encoder f(.) + projection head g(.). Loss is on g(f(x))."""
    def __init__(self, in_dim, hid=64, rep_dim=32, proj_dim=16):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(in_dim, hid), nn.ReLU(), nn.Linear(hid, rep_dim))
        self.proj = nn.Sequential(
            nn.Linear(rep_dim, proj_dim), nn.ReLU(), nn.Linear(proj_dim, proj_dim))
    def representation(self, x):
        return self.backbone(x)            # h = f(x): kept for downstream
    def forward(self, x):
        return self.proj(self.backbone(x)) # z = g(f(x)): used for the loss

## 2. Demonstrate the benefit (no-labels contrastive toy task)
K latent identities; an augmentation = anchor + noise + scale + feature mask (two correlated views = a positive pair). Train with NT-Xent, NO labels. Then check same-id vs different-id similarity.

In [ ]:
def augment(anchors, idx, noise=0.15):
    a = anchors[idx]
    a = a + noise * torch.randn_like(a)
    scale = 0.8 + 0.4 * torch.rand(a.shape[0], 1)
    mask = (torch.rand_like(a) > 0.1).float()
    return a * scale * mask

def mean_same_diff_sim(model, anchors, n_per=20):
    model.eval()
    K = anchors.shape[0]
    with torch.no_grad():
        ids = torch.arange(K).repeat_interleave(n_per)
        h = F.normalize(model.representation(augment(anchors, ids)), dim=1)
        S = h @ h.t()
        same = ids.unsqueeze(0) == ids.unsqueeze(1)
        off = ~torch.eye(len(ids), dtype=torch.bool)
        return S[same & off].mean().item(), S[~same].mean().item()

torch.manual_seed(0)
in_dim, K = 24, 8
common = torch.randn(in_dim)
anchors = 0.6 * torch.randn(K, in_dim) + common   # identities overlap -> learning is visible

model = Encoder(in_dim)
s0, d0 = mean_same_diff_sim(model, anchors)        # BEFORE training

opt = torch.optim.Adam(model.parameters(), lr=2e-3)
model.train()
for step in range(800):
    idx = torch.randint(0, K, (64,))              # random identities, NO labels in loss
    z1, z2 = model(augment(anchors, idx)), model(augment(anchors, idx))
    loss = nt_xent_loss(z1, z2, temperature=0.5)
    opt.zero_grad(); loss.backward(); opt.step()

s1, d1 = mean_same_diff_sim(model, anchors)        # AFTER training
print(f"BEFORE  same {s0:+.3f}  diff {d0:+.3f}  gap {s0-d0:+.3f}")
print(f"AFTER   same {s1:+.3f}  diff {d1:+.3f}  gap {s1-d1:+.3f}")

## 3. Sanity checks

In [ ]:
# 1) embeddings L2-normalized
zn = F.normalize(torch.randn(10, 16), dim=1)
assert torch.allclose(zn.norm(dim=1), torch.ones(10), atol=1e-5)

# 2) loss identifies positives (perfect pairs -> ~0)
N, D = 8, 16
base = torch.eye(N, D)
aligned = nt_xent_loss(base.clone(), base.clone(), 0.1).item()
rand_l = nt_xent_loss(torch.randn(N, D), torch.randn(N, D), 0.1).item()
assert aligned < 0.1 and aligned < rand_l

# 3) similarity matrix excludes self-pairs (diagonal masked)
z = F.normalize(torch.randn(2 * N, D), dim=1)
sim = (z @ z.t() / 0.5).masked_fill(torch.eye(2 * N, dtype=torch.bool), float('-inf'))
assert torch.isinf(torch.diagonal(sim)).all()

# 4) temperature: lower T -> sharper
lg = torch.tensor([1.0, 0.5, 0.2, -0.3])
assert F.softmax(lg / 0.1, 0).max() > F.softmax(lg / 1.0, 0).max()

# 5) after training, same-id sim > diff-id (the demonstration)
s1, d1 = mean_same_diff_sim(model, anchors)
assert s1 > d1 + 0.2

# 6) gradient flows to the encoder backbone
m = Encoder(24); anc = torch.randn(8, 24); idx = torch.arange(4)
nt_xent_loss(m(augment(anc, idx)), m(augment(anc, idx)), 0.5).backward()
assert m.backbone[0].weight.grad.abs().sum().item() > 0

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
def nt_xent_solution(z1, z2, temperature=0.5):
    N = z1.shape[0]
    z = torch.cat([z1, z2], dim=0)
    z = F.normalize(z, dim=1)
    sim = z @ z.t() / temperature
    sim = sim.masked_fill(torch.eye(2 * N, dtype=torch.bool, device=z.device), float('-inf'))
    targets = torch.cat([torch.arange(N, 2 * N), torch.arange(0, N)]).to(z.device)
    return F.cross_entropy(sim, targets)